In [ ]:
# Copyright 2026 Google LLC
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#     https://www.apache.org/licenses/LICENSE-2.0
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# 01 · Setup

Enable the managed BigQuery MCP server, set up Application Default Credentials,
and confirm both halves are reachable: the MCP endpoint and Claude on Vertex.

This is the foundation for the bridge built in notebooks 03 and 04 — remember
that Vertex has no native MCP connector, so reachable endpoints are step one.

Cells are paste-ready for Cloud Shell.

## Load configuration from .env

Every value comes from `.env` (copy it from `.env.example`). Nothing is
hardcoded; this cell fails loudly if a variable is missing.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

REQUIRED = [
    "GOOGLE_CLOUD_PROJECT",
    "GOOGLE_CLOUD_REGION",
    "CLAUDE_MODEL_ID",
    "BQ_MCP_ENDPOINT",
    "BQ_DATASET",
]
missing = [k for k in REQUIRED if not os.environ.get(k)]
assert not missing, f"Missing in .env (see .env.example): {missing}"

PROJECT = os.environ["GOOGLE_CLOUD_PROJECT"]
REGION = os.environ["GOOGLE_CLOUD_REGION"]
MODEL_ID = os.environ["CLAUDE_MODEL_ID"]
MCP_ENDPOINT = os.environ["BQ_MCP_ENDPOINT"]
DATASET = os.environ["BQ_DATASET"]
print("Config loaded. Project:", PROJECT, "| Region:", REGION, "| Model:", MODEL_ID)

## Enable the BigQuery API and the MCP surface, grant IAM roles

The MCP server uses OAuth2/IAM and rejects API keys. The calling identity needs
`bigquery.user`, `bigquery.dataViewer`, and `mcp.toolUser`.

In [ ]:
!gcloud services enable bigquery.googleapis.com --project={PROJECT}
!gcloud beta services mcp enable bigquery.googleapis.com --project={PROJECT}

ACCOUNT = !gcloud config get-value account
ACCOUNT = ACCOUNT[0].strip()
for role in ("roles/bigquery.user", "roles/bigquery.dataViewer", "roles/mcp.toolUser"):
    !gcloud projects add-iam-policy-binding {PROJECT} --member=user:{ACCOUNT} --role={role} --condition=None

## Set up Application Default Credentials

Both the MCP server and Claude on Vertex authenticate via ADC. In Cloud Shell
this opens an interactive consent flow.

In [ ]:
!gcloud auth application-default login

## Smoke check A — the MCP endpoint is reachable

A lightweight check that the endpoint resolves and accepts an authenticated
request. The full MCP handshake (`initialize` + `tools/list`) is exercised in
notebook 02; here we only confirm reachability and that the bearer token is
accepted.

In [ ]:
import json
import subprocess
import urllib.request
import urllib.error

# Use the ADC (cloud-platform-scoped) token: the MCP gateway accepts a plain
# access token for metadata, but tool execution runs a BigQuery job that needs
# this scope. See the "IAM roles" gotcha in the README.
token = subprocess.check_output(
    ["gcloud", "auth", "application-default", "print-access-token"], text=True
).strip()
body = json.dumps({"jsonrpc": "2.0", "id": 1, "method": "tools/list"}).encode()
req = urllib.request.Request(
    MCP_ENDPOINT,
    data=body,
    headers={
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
        "Accept": "application/json, text/event-stream",
    },
    method="POST",
)
try:
    with urllib.request.urlopen(req) as resp:
        print("MCP endpoint reachable. HTTP", resp.status)
except urllib.error.HTTPError as e:
    # A 4xx still proves the endpoint resolved and processed an authenticated request.
    print("MCP endpoint reachable. HTTP", e.code, e.reason)

## Smoke check B — Claude on Vertex responds

`AnthropicVertex` authenticates through ADC; no API key is used. `region` may be
`global` (recommended) or a specific region.

In [ ]:
from anthropic import AnthropicVertex

client = AnthropicVertex(project_id=PROJECT, region=REGION)
msg = client.messages.create(
    model=MODEL_ID,
    max_tokens=16,
    messages=[{"role": "user", "content": "Reply with exactly: setup OK"}],
)
print("Claude on Vertex replied:", "".join(b.text for b in msg.content if b.type == "text"))

## Smoke check C — BigQuery public data is queryable

Confirms the project can run a BigQuery job against the public dataset used
throughout the tutorial.

In [ ]:
!bq query --project_id={PROJECT} --use_legacy_sql=false --format=pretty 'SELECT COUNT(*) AS rows FROM `{DATASET}.usa_names.usa_1910_2013`'

## Recap

If all three smoke checks passed, the foundation is in place: the MCP endpoint
is reachable, Claude on Vertex responds, and BigQuery is queryable. Next,
notebook 02 exercises each half on its own.

## Cleanup

This notebook makes **persistent project changes** (enables APIs, grants IAM
roles to your user) rather than per-run resources, so there is nothing to tear
down between runs. To fully revert (optional):

```bash
# remove the role bindings
for role in roles/bigquery.user roles/bigquery.dataViewer roles/mcp.toolUser; do
  gcloud projects remove-iam-policy-binding "$GOOGLE_CLOUD_PROJECT" \
    --member="user:$(gcloud config get-value account)" --role="$role" --condition=None
done
# (APIs can be left enabled; disable with `gcloud services disable` if desired)
```